![clothing_classification](clothing_classification.png)


Fashion Forward is a new AI-based e-commerce clothing retailer.
They want to use image classification to automatically categorize new product listings, making it easier for customers to find what they're looking for. It will also assist in inventory management by quickly sorting items.

As a data scientist tasked with implementing a garment classifier, your primary objective is to develop a machine learning model capable of accurately categorizing images of clothing items into distinct garment types such as shirts, trousers, shoes, etc.

In [7]:
# Run the cells below first

In [8]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchmetrics import Accuracy, Precision, Recall

In [9]:
# Load datasets
from torchvision import datasets
import torchvision.transforms as transforms

train_data = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
test_data = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transforms.ToTensor())

In [10]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchmetrics.functional import accuracy as acc_fn, precision as prec_fn, recall as rec_fn

# ======================
# 1. Load FashionMNIST
# ======================
transform = transforms.Compose([transforms.ToTensor()])
train_data = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=1000, shuffle=False)

# ======================
# 2. Define CNN model
# ======================
class CNNClassifier(nn.Module):
    def __init__(self):
        super(CNNClassifier, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3)
        self.dropout = nn.Dropout(0.25)

        # We'll compute the flattened size dynamically
        self._to_linear = None
        self.convs(torch.randn(1, 1, 28, 28))  # run once to set _to_linear

        self.fc1 = nn.Linear(self._to_linear, 128)
        self.fc2 = nn.Linear(128, 10)

    def convs(self, x):
        x = torch.relu(self.conv1(x))
        x = self.pool(x)
        x = torch.relu(self.conv2(x))
        x = self.pool(x)
        if self._to_linear is None:
            self._to_linear = x.numel()
        return x

    def forward(self, x):
        x = self.convs(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# ======================
# 3. Training setup
# ======================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNNClassifier().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# ======================
# 4. Train for 1–2 epochs
# ======================
epochs = 2
model.train()
for epoch in range(epochs):
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}")

# ======================
# 5. Evaluate on test set
# ======================
model.eval()
predictions = []
true_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        predictions.extend(preds.cpu().numpy())
        true_labels.extend(labels.numpy())

predictions = list(predictions)

# ======================
# 6. Compute metrics (fixed)
# ======================
y_true = torch.tensor(true_labels)
y_pred = torch.tensor(predictions)

accuracy = acc_fn(y_pred, y_true, task="multiclass", num_classes=10).item()
precision = prec_fn(y_pred, y_true, task="multiclass", num_classes=10, average=None).tolist()
recall = rec_fn(y_pred, y_true, task="multiclass", num_classes=10, average=None).tolist()

# ======================
# 7. Display results
# ======================
print(f"\n✅ Accuracy: {accuracy:.4f}")
print("Precision per class:", [round(p, 4) for p in precision])
print("Recall per class:", [round(r, 4) for r in recall])



Epoch 1/2, Loss: 0.5356
Epoch 2/2, Loss: 0.3583

✅ Accuracy: 0.8704
Precision per class: [0.764, 0.9888, 0.6799, 0.9151, 0.8244, 0.971, 0.7335, 0.9492, 0.9815, 0.9453]
Recall per class: [0.903, 0.97, 0.911, 0.851, 0.709, 0.971, 0.534, 0.934, 0.954, 0.967]
